[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/10_moe_and_routing.ipynb)

# 10. Mixture-of-Experts and routing

router logits에서 top-k 선택, gather, expert computation, scatter까지 직접 본다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Dense FFN baseline

모든 token이 같은 FFN을 지난다.


In [ ]:
tokens = torch.randn(4, 8, device=device)
dense = nn.Sequential(nn.Linear(8, 16), nn.SiLU(), nn.Linear(16, 8)).to(device)

print(dense(tokens).shape)


In [ ]:
_ = profile_call("dense FFN", dense, tokens)


## 2. Soft routing

모든 expert 출력을 router 확률로 혼합한다.


In [ ]:
experts = nn.ModuleList([nn.Linear(8, 8, bias=False).to(device) for _ in range(3)])
router = nn.Linear(8, 3, bias=False).to(device)

weights = router(tokens).softmax(-1)
expert_out = torch.stack([e(tokens) for e in experts], dim=1)
soft_out = (weights[..., None] * expert_out).sum(dim=1)

print("router weights:\n", weights)


In [ ]:
def soft_routing_once():
    weights_ = router(tokens).softmax(-1)
    expert_out_ = torch.stack([expert(tokens) for expert in experts], dim=1)
    return (weights_[..., None] * expert_out_).sum(dim=1)

_ = profile_call("soft routing", soft_routing_once)


## 3. Top-1 routing

token마다 expert 하나만 선택한다.


In [ ]:
choice = router(tokens).argmax(-1)
out = torch.empty_like(tokens)

for expert_id, expert in enumerate(experts):
    token_ids = (choice == expert_id).nonzero(as_tuple=True)[0]
    if token_ids.numel() > 0:
        out[token_ids] = expert(tokens[token_ids])

print("choice:", choice)
print("out shape:", out.shape)


In [ ]:
_ = profile_call("router top1 select", lambda: router(tokens).argmax(-1))


## 4. Top-2 routing

상위 두 expert를 선택하고 gate로 합친다.


In [ ]:
logits = router(tokens)
topv, topi = logits.topk(2, dim=-1)
gate = topv.softmax(-1)

print("top2 expert ids:\n", topi)
print("top2 gates:\n", gate)


In [ ]:
_ = profile_call("top2 routing", lambda: router(tokens).topk(2, dim=-1))


## 5. Shared expert

항상 실행되는 shared path와 routed path를 합친다.


In [ ]:
shared = nn.Linear(8, 8, bias=False).to(device)
shared_out = shared(tokens)
combined = shared_out + out

print("shared + routed:", combined.shape)


In [ ]:
_ = profile_call("shared expert", lambda z: shared(z) + out, tokens)


## References and provenance

**[10.1] Switch Transformer**
- 출처: Fedus et al., Switch Transformers
- 이 노트북에서 가져온 부분: top-1 sparse routing

**[10.2] Mixtral**
- 출처: Mistral AI Mixtral report
- 이 노트북에서 가져온 부분: top-2 sparse MoE

**[10.3] DeepSeekMoE**
- 출처: DeepSeekMoE / DeepSeek-V2/V3 papers
- 이 노트북에서 가져온 부분: fine-grained experts and shared experts

**[10.4] Latent MoE lineage**
- 출처: recent Kimi-family MoE reports
- 이 노트북에서 가져온 부분: compressed/latent routing variants
